In [2]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt 

In [3]:
# 1. LOAD DATA
df = pd.read_csv("Groceries_dataset.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nMissing values:\n", df.isnull().sum())
print("\nUnique members:", df['Member_number'].nunique())
print("Unique items:", df['itemDescription'].nunique())

Shape: (38765, 3)

First 5 rows:
   Member_number        Date   itemDescription
0           1808  21-07-2015    tropical fruit
1           2552  05-01-2015        whole milk
2           2300  19-09-2015         pip fruit
3           1187  12-12-2015  other vegetables
4           3037  01-02-2015        whole milk

Missing values:
 Member_number      0
Date               0
itemDescription    0
dtype: int64

Unique members: 3898
Unique items: 167


In [4]:
# 2. GROUP INTO TRANSACTIONS
# Each unique (Member_number, Date) combo = one "shopping basket"
df['Transaction_ID'] = df['Member_number'].astype(str) + "_" + df['Date'].astype(str)

transactions = df.groupby('Transaction_ID')['itemDescription'].apply(list).tolist()

print("\nTotal transactions:", len(transactions))
print("Sample transaction:", transactions[0])



Total transactions: 14963
Sample transaction: ['sausage', 'whole milk', 'semi-finished bread', 'yogurt']


In [5]:
# 3. ENCODE TRANSACTIONS (ONE-HOT FORMAT)
# Apriori needs a table where each row = transaction, each column = item (True/False)
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

print("\nOne-hot encoded basket shape:", basket_df.shape)


One-hot encoded basket shape: (14963, 167)


In [6]:

# 4. APPLY APRIORI ALGORITHM
# min_support = 0.01 means the itemset must appear in at least 1% of transactions
frequent_itemsets = apriori(basket_df, min_support=0.001, use_colnames=True)
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)

print("\nTop 10 frequent itemsets:")
print(frequent_itemsets.head(10))


Top 10 frequent itemsets:
      support                       itemsets
146  0.157923        frozenset({whole milk})
90   0.122101  frozenset({other vegetables})
109  0.110005        frozenset({rolls/buns})
123  0.097106              frozenset({soda})
147  0.085879            frozenset({yogurt})
110  0.069572   frozenset({root vegetables})
139  0.067767    frozenset({tropical fruit})
10   0.060683     frozenset({bottled water})
115  0.060349           frozenset({sausage})
28   0.053131      frozenset({citrus fruit})


In [7]:
# 5. GENERATE ASSOCIATION RULES
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules.sort_values(by='lift', ascending=False)

print("\nTop 10 association rules (by lift):")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


Top 10 association rules (by lift):
                          antecedents                       consequents  \
81    frozenset({yogurt, whole milk})              frozenset({sausage})   
84               frozenset({sausage})   frozenset({yogurt, whole milk})   
82   frozenset({sausage, whole milk})               frozenset({yogurt})   
83                frozenset({yogurt})  frozenset({sausage, whole milk})   
114  frozenset({specialty chocolate})         frozenset({citrus fruit})   
115         frozenset({citrus fruit})  frozenset({specialty chocolate})   
80       frozenset({yogurt, sausage})           frozenset({whole milk})   
85            frozenset({whole milk})      frozenset({yogurt, sausage})   
190       frozenset({tropical fruit})                frozenset({flour})   
191                frozenset({flour})       frozenset({tropical fruit})   

      support  confidence      lift  
81   0.001470    0.131737  2.182917  
84   0.001470    0.024363  2.182917  
82   0.001470    0.1641

In [8]:

# 6. SAVE RESULTS
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_csv(
    "association_rules.csv", index=False
)
print("\nSaved: association_rules.csv")


Saved: association_rules.csv


In [9]:

# 7. VISUALIZATION
import matplotlib.pyplot as plt

top_rules = rules.head(10).copy()
top_rules['rule'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x))) + \
                     " -> " + top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

plt.figure(figsize=(9, 6))
plt.barh(top_rules['rule'], top_rules['lift'], color='teal')
plt.xlabel("Lift")
plt.title("Top 10 Association Rules by Lift")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("top_rules_lift.png")
plt.close()

print("Saved: top_rules_lift.png")

Saved: top_rules_lift.png
